# 05. Avaliação residual

Depois do [`04_atribuir.ipynb`](04_atribuir.ipynb). Não mexe no greedy operacional
(corte `THRESHOLD_AVALIACAO`, default 0,99). Três recortes no parquet de pares
(`predict` do 02b, em geral ≥ 0,5):

1. Censos cujo melhor score é **menor que 0,99**: quantos seriam 1_para_1 se o
   corte fosse 0,90 ou 0,95 (exatamente 1 CPF ≥ T, e esse CPF não é candidato de
   outro Censo no mesmo T). Sem reclustering.
2. IDs em clusters que **não** são `1_para_1` (`TIPOS_CLUSTER_RESIDUAL`,
   default `1_cpf_n_censo` e `outros`). Volta à **lista de pares do parquet**
   (não só arestas internas do cluster em 0,99). Melhor CPF por Censo; depois,
   nos CPFs ainda disputados, o Censo de maior score. Isto é “melhor por Censo
   e drop de colisão”, só neste residual. `MIN_N_CLUSTER` (default 2) corta
   tamanho; dá para subir se quiser só os grandes.
3. Pares com `p < 0,99`, `primeiro_nome_phon` igual e `data_nascimento` igual.

Pré-requisito: 02b (predictions + clusters), 00b (limpos), 04 (`splink_atribuicao.parquet`).


In [ ]:
import sys
from pathlib import Path

PROB_DIR = Path.cwd()
if PROB_DIR.name == 'notebooks':
    PROB_DIR = PROB_DIR.parent
if str(PROB_DIR) not in sys.path:
    sys.path.insert(0, str(PROB_DIR))

import pandas as pd
from IPython.display import display

from config import (
    COHORT_DEDUP_ARQUIVO,
    METRICAS_RESIDUAL,
    SPLINK_ATRIBUICAO,
    SPLINK_CLUSTERS,
    SPLINK_INPUT_VIEW,
    SPLINK_PREDICTIONS,
    TABELA_CENSO_LIMPA,
    TABELA_CPF_LIMPA,
    THRESHOLD_AVALIACAO,
    drop_splink_temp_tables,
    get_connection,
    materialize_cluster_composicao,
    materialize_gt_no_subset,
    materialize_splink_input,
    print_paths,
    require_input,
    require_tables,
)

T_90 = 0.90
T_95 = 0.95
# Clusters ≠ 1_para_1. Singleton fica de fora (n=1). Ajuste aqui se quiser
# só `outros`, só `1_cpf_n_censo`, ou subir MIN_N_CLUSTER para os grandes.
TIPOS_CLUSTER_RESIDUAL = ('1_cpf_n_censo', 'outros')
MIN_N_CLUSTER = 2
TOP_N = 30


def display_metricas(df_uma_linha):
    """Uma linha de capa: inteiros com milhar, o resto com 4 casas."""
    row = df_uma_linha.iloc[0]
    linhas = []
    for k, v in row.items():
        if v is None or (isinstance(v, float) and pd.isna(v)):
            txt = ''
        elif isinstance(v, bool):
            txt = v
        elif pd.api.types.is_number(v) and not isinstance(v, bool):
            fv = float(v)
            if abs(fv - round(fv)) < 1e-12 and abs(fv) >= 1:
                txt = f'{int(round(fv)):,}'
            else:
                txt = f'{fv:.4f}'
        else:
            txt = v
        linhas.append({'metrica': k, 'valor': txt})
    display(pd.DataFrame(linhas))


print_paths()
require_input(COHORT_DEDUP_ARQUIVO, label='COHORT')
require_input(SPLINK_PREDICTIONS, label='SPLINK_PREDICTIONS (rode o 02b_aplicar antes)')
require_input(SPLINK_CLUSTERS, label='SPLINK_CLUSTERS (rode o 02b_aplicar antes)')
require_input(SPLINK_ATRIBUICAO, label='SPLINK_ATRIBUICAO (rode o 04_atribuir antes)')

con = get_connection()
drop_splink_temp_tables(con)
require_tables(con, [TABELA_CENSO_LIMPA, TABELA_CPF_LIMPA], notebook_origem='00b')
materialize_splink_input(con)
print('Threshold operacional:', THRESHOLD_AVALIACAO)
print('Cortes residuais:', T_90, T_95)
print('TIPOS_CLUSTER_RESIDUAL:', TIPOS_CLUSTER_RESIDUAL)
print('MIN_N_CLUSTER:', MIN_N_CLUSTER)


## 0. Predictions, clusters, atribuição, ouro

`unique_id_l` = Censo. `atribuicao_censo_cpf` é a lista do 04 (não recalcula greedy).


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE splink_predictions AS
SELECT
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_l
        ELSE unique_id_r
    END AS unique_id_l,
    CASE
        WHEN unique_id_l LIKE 'censo_%' THEN unique_id_r
        ELSE unique_id_l
    END AS unique_id_r,
    match_probability
FROM read_parquet('{SPLINK_PREDICTIONS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE splink_clusters AS
SELECT * FROM read_parquet('{SPLINK_CLUSTERS}')
''')

con.execute(f'''
CREATE OR REPLACE TABLE atribuicao_censo_cpf AS
SELECT * FROM read_parquet('{SPLINK_ATRIBUICAO}')
''')

materialize_cluster_composicao(con)
display(con.execute('''
SELECT tipo, CAST(COUNT(*) AS BIGINT) AS n_clusters,
       CAST(SUM(n) AS BIGINT) AS n_registros,
       CAST(SUM(n_censo) AS BIGINT) AS n_censo,
       CAST(SUM(n_cpf) AS BIGINT) AS n_cpf
FROM cluster_composicao
GROUP BY tipo
ORDER BY n_registros DESC
''').df())

counts = materialize_gt_no_subset(con, cohort_parquet=COHORT_DEDUP_ARQUIVO)
n_gt = counts['n_gt_no_subset']
print('Ouro 1:1 no subset:', f'{n_gt:,}')
print('Atribuidos no 04:', con.execute(
    'SELECT COUNT(*) FROM atribuicao_censo_cpf'
).fetchone()[0])


## 1. Únicos 1-1 em 0,90 e 0,95 (abaixo de 0,99)

População: Censos cujo **melhor** `match_probability` no parquet é `< 0,99`.
Quem já tem par ≥ 0,99 não entra.

Em cada corte T: candidatos = pares com `p ≥ T`. Censo único = exatamente 1 CPF;
CPF único = exatamente 1 Censo (em **toda** a lista nesse T, não só nos abaixo
de 0,99). 1_para_1 = os dois únicos.


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE censo_max_p AS
SELECT unique_id_l AS unique_id_censo, MAX(match_probability) AS pmax
FROM splink_predictions
GROUP BY 1
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_abaixo_099 AS
SELECT unique_id_censo, pmax
FROM censo_max_p
WHERE pmax < {THRESHOLD_AVALIACAO}
''')

attrs = f'''
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf
'''


def tab_unicos(t: float) -> str:
    return 'unicos_' + f'{t:.2f}'.replace('.', '_')


def unicos_1a1(t: float) -> pd.DataFrame:
    tab = tab_unicos(t)
    con.execute(f'''
    CREATE OR REPLACE TABLE {tab} AS
    WITH cand AS (
        SELECT unique_id_l, unique_id_r, match_probability
        FROM splink_predictions
        WHERE match_probability >= {t}
    ),
    censo_n AS (
        SELECT unique_id_l, COUNT(DISTINCT unique_id_r) AS n_cpf
        FROM cand GROUP BY 1
    ),
    cpf_n AS (
        SELECT unique_id_r, COUNT(DISTINCT unique_id_l) AS n_censo
        FROM cand GROUP BY 1
    )
    SELECT
        c.unique_id_l AS unique_id_censo,
        c.unique_id_r AS unique_id_cpf,
        c.match_probability
    FROM cand c
    JOIN censo_n cn ON cn.unique_id_l = c.unique_id_l AND cn.n_cpf = 1
    JOIN cpf_n pn ON pn.unique_id_r = c.unique_id_r AND pn.n_censo = 1
    JOIN censo_abaixo_099 a ON a.unique_id_censo = c.unique_id_l
    ''')
    return con.execute(f'''
    SELECT
        CAST(COUNT(*) AS BIGINT) AS n_1_para_1,
        CAST(COALESCE(SUM(CASE WHEN gt.unique_id_censo IS NOT NULL THEN 1 ELSE 0 END), 0) AS BIGINT)
            AS n_na_ouro,
        CAST(COALESCE(SUM(CASE WHEN u.unique_id_cpf = gt.unique_id_cpf THEN 1 ELSE 0 END), 0) AS BIGINT)
            AS n_cpf_certo
    FROM {tab} u
    LEFT JOIN gt_no_subset gt ON gt.unique_id_censo = u.unique_id_censo
    ''').df()


n_abaixo = con.execute(
    'SELECT CAST(COUNT(*) AS BIGINT) FROM censo_abaixo_099'
).fetchone()[0]
capa_090 = unicos_1a1(T_90)
capa_095 = unicos_1a1(T_95)
capa1 = pd.DataFrame([{
    'n_censo_abaixo_099': n_abaixo,
    'n_1_para_1_090': int(capa_090['n_1_para_1'].iloc[0]),
    'n_na_ouro_090': int(capa_090['n_na_ouro'].iloc[0]),
    'n_cpf_certo_090': int(capa_090['n_cpf_certo'].iloc[0]),
    'n_1_para_1_095': int(capa_095['n_1_para_1'].iloc[0]),
    'n_na_ouro_095': int(capa_095['n_na_ouro'].iloc[0]),
    'n_cpf_certo_095': int(capa_095['n_cpf_certo'].iloc[0]),
}])
display_metricas(capa1)

print(f'Amostra 1_para_1 em {T_95} (abaixo de {THRESHOLD_AVALIACAO}):')
display(con.execute(f'''
SELECT
    u.unique_id_censo, u.unique_id_cpf, u.match_probability,
    {attrs}
FROM {tab_unicos(T_95)} u
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = u.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = u.unique_id_cpf
ORDER BY u.match_probability DESC
LIMIT {TOP_N}
''').df())


## 2. Clusters ≠ 1_para_1: pares do parquet, duas passagens

População: `tipo` em `TIPOS_CLUSTER_RESIDUAL` e `n ≥ MIN_N_CLUSTER`. Default:
`1_cpf_n_censo` e `outros`, tamanho ≥ 2 (não só o mega). A capa por tipo
embaixo é o recorte para decidir o que entra.

Arestas = **todos** os pares do parquet desses Censos (não só o cluster em 0,99).

Passagem 1: melhor CPF por Censo. Conta CPFs ainda com mais de um Censo.
Passagem 2: nesses, fica o Censo de maior score. Quem perde o CPF sai da lista.

Isto é “melhor CPF por Censo e depois drop de colisão”, de propósito só no residual.


In [ ]:
tipos_sql = ', '.join(f"'{t}'" for t in TIPOS_CLUSTER_RESIDUAL)

print('Clusters ≠ 1_para_1 (todos os tamanhos), para decidir o recorte:')
display(con.execute('''
SELECT tipo,
       CAST(COUNT(*) AS BIGINT) AS n_clusters,
       CAST(SUM(n) AS BIGINT) AS n_registros,
       CAST(SUM(n_censo) AS BIGINT) AS n_censo,
       CAST(SUM(n_cpf) AS BIGINT) AS n_cpf,
       CAST(MAX(n) AS BIGINT) AS n_max
FROM cluster_composicao
WHERE tipo <> '1_para_1'
GROUP BY tipo
ORDER BY n_registros DESC
''').df())

display(con.execute(f'''
SELECT cluster_id, tipo, CAST(n AS BIGINT) AS n,
       CAST(n_censo AS BIGINT) AS n_censo, CAST(n_cpf AS BIGINT) AS n_cpf
FROM cluster_composicao
WHERE tipo IN ({tipos_sql})
ORDER BY n DESC
LIMIT 15
''').df())

con.execute(f'''
CREATE OR REPLACE TABLE clusters_residual AS
SELECT cluster_id, tipo, n, n_censo, n_cpf
FROM cluster_composicao
WHERE tipo IN ({tipos_sql}) AND n >= {MIN_N_CLUSTER}
''')

con.execute(f'''
CREATE OR REPLACE TABLE censo_residual AS
SELECT s.unique_id AS unique_id_censo, sc.cluster_id, c.tipo
FROM splink_clusters sc
JOIN clusters_residual c ON c.cluster_id = sc.cluster_id
JOIN {SPLINK_INPUT_VIEW} s ON s.unique_id = sc.unique_id
WHERE s.origem = 'censo'
''')

con.execute('''
CREATE OR REPLACE TABLE pares_residual AS
SELECT p.unique_id_l, p.unique_id_r, p.match_probability
FROM splink_predictions p
JOIN censo_residual c ON c.unique_id_censo = p.unique_id_l
''')

con.execute('''
CREATE OR REPLACE TABLE residual_pass1 AS
SELECT unique_id_l AS unique_id_censo, unique_id_r AS unique_id_cpf, match_probability
FROM (
    SELECT *,
        row_number() OVER (
            PARTITION BY unique_id_l
            ORDER BY match_probability DESC, unique_id_r
        ) AS rn
    FROM pares_residual
)
WHERE rn = 1
''')

con.execute('''
CREATE OR REPLACE TABLE residual_pass2 AS
SELECT unique_id_censo, unique_id_cpf, match_probability
FROM (
    SELECT *,
        row_number() OVER (
            PARTITION BY unique_id_cpf
            ORDER BY match_probability DESC, unique_id_censo
        ) AS rn
    FROM residual_pass1
)
WHERE rn = 1
''')

capa2 = con.execute('''
SELECT
    CAST((SELECT COUNT(*) FROM clusters_residual) AS BIGINT) AS n_clusters_residual,
    CAST((SELECT COUNT(*) FROM censo_residual) AS BIGINT) AS n_censo_residual,
    CAST((SELECT COUNT(*) FROM residual_pass1) AS BIGINT) AS n_apos_pass1,
    CAST((
        SELECT COUNT(*) FROM (
            SELECT unique_id_cpf FROM residual_pass1
            GROUP BY 1 HAVING COUNT(*) > 1
        )
    ) AS BIGINT) AS n_cpf_disputado,
    CAST((SELECT COUNT(*) FROM residual_pass2) AS BIGINT) AS n_1a1_pass2,
    CAST(COALESCE((
        SELECT SUM(CASE WHEN m.unique_id_cpf = gt.unique_id_cpf THEN 1 ELSE 0 END)
        FROM residual_pass2 m
        JOIN gt_no_subset gt ON gt.unique_id_censo = m.unique_id_censo
    ), 0) AS BIGINT) AS n_ouro_cpf_certo
''').df()
display_metricas(capa2)

print('Capa por tipo (recorte para decidir):')
display(con.execute('''
SELECT
    c.tipo,
    CAST(COUNT(DISTINCT c.cluster_id) AS BIGINT) AS n_clusters,
    CAST(COUNT(*) AS BIGINT) AS n_censo,
    CAST(COUNT(p1.unique_id_censo) AS BIGINT) AS n_apos_pass1,
    CAST(COUNT(p2.unique_id_censo) AS BIGINT) AS n_1a1_pass2
FROM censo_residual c
LEFT JOIN residual_pass1 p1 ON p1.unique_id_censo = c.unique_id_censo
LEFT JOIN residual_pass2 p2 ON p2.unique_id_censo = c.unique_id_censo
GROUP BY c.tipo
ORDER BY n_censo DESC
''').df())

print('Amostra 1a1 após passagem 2:')
display(con.execute(f'''
SELECT
    m.unique_id_censo, m.unique_id_cpf, m.match_probability,
    c.tipo, c.cluster_id,
    {attrs}
FROM residual_pass2 m
JOIN censo_residual c ON c.unique_id_censo = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = m.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = m.unique_id_cpf
ORDER BY m.match_probability DESC
LIMIT {TOP_N}
''').df())


## 3. Primeiro nome e DOB iguais, `p < 0,99`

Todos os pares do parquet abaixo do corte operacional em que
`primeiro_nome_phon` é igual (ambos não nulos) e `data_nascimento` é igual
(`IS NOT DISTINCT FROM`, ambos não nulos).


In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE pares_nome_dob AS
SELECT
    p.unique_id_l AS unique_id_censo,
    p.unique_id_r AS unique_id_cpf,
    p.match_probability,
    (a.unique_id_censo IS NOT NULL) AS ja_atribuido_04
FROM splink_predictions p
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = p.unique_id_l
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = p.unique_id_r
LEFT JOIN atribuicao_censo_cpf a ON a.unique_id_censo = p.unique_id_l
WHERE p.match_probability < {THRESHOLD_AVALIACAO}
  AND ca.primeiro_nome_phon IS NOT NULL
  AND pb.primeiro_nome_phon IS NOT NULL
  AND ca.primeiro_nome_phon = pb.primeiro_nome_phon
  AND ca.data_nascimento IS NOT NULL
  AND pb.data_nascimento IS NOT NULL
  AND ca.data_nascimento IS NOT DISTINCT FROM pb.data_nascimento
''')

capa3 = con.execute(f'''
SELECT
    CAST(COUNT(*) AS BIGINT) AS n_nome_dob_abaixo_099,
    CAST(COALESCE(SUM(CASE WHEN match_probability >= {T_95} THEN 1 ELSE 0 END), 0) AS BIGINT)
        AS n_p_ge_095,
    CAST(COALESCE(SUM(CASE
        WHEN match_probability >= {T_90} AND match_probability < {T_95}
        THEN 1 ELSE 0
    END), 0) AS BIGINT) AS n_p_090_095,
    CAST(COALESCE(SUM(CASE WHEN match_probability < {T_90} THEN 1 ELSE 0 END), 0) AS BIGINT)
        AS n_p_lt_090,
    CAST(COALESCE(SUM(CASE WHEN ja_atribuido_04 THEN 1 ELSE 0 END), 0) AS BIGINT)
        AS n_censo_ja_atribuido_04
FROM pares_nome_dob
''').df()
display_metricas(capa3)

print('Amostra primeiro nome + DOB iguais, p < 0,99:')
display(con.execute(f'''
SELECT
    d.unique_id_censo, d.unique_id_cpf, d.match_probability, d.ja_atribuido_04,
    ca.nome_completo AS nome_censo,
    pb.nome_completo AS nome_cpf,
    ca.nome_completo_phon AS nome_phon_censo,
    pb.nome_completo_phon AS nome_phon_cpf,
    ca.primeiro_nome_phon AS primeiro_censo,
    pb.primeiro_nome_phon AS primeiro_cpf,
    ca.sexo AS sexo_censo,
    pb.sexo AS sexo_cpf,
    ca.data_nascimento AS dob_censo,
    pb.data_nascimento AS dob_cpf,
    ca.idade AS idade_censo,
    pb.idade AS idade_cpf
FROM pares_nome_dob d
JOIN {SPLINK_INPUT_VIEW} ca ON ca.unique_id = d.unique_id_censo
JOIN {SPLINK_INPUT_VIEW} pb ON pb.unique_id = d.unique_id_cpf
ORDER BY d.match_probability DESC
LIMIT {TOP_N}
''').df())


In [ ]:
metricas = capa1.copy()
for col in capa2.columns:
    metricas[col] = capa2[col].iloc[0]
for col in capa3.columns:
    metricas[col] = capa3[col].iloc[0]
metricas['threshold_avaliacao'] = THRESHOLD_AVALIACAO
metricas['min_n_cluster'] = MIN_N_CLUSTER
metricas['tipos_cluster_residual'] = ','.join(TIPOS_CLUSTER_RESIDUAL)
metricas.to_csv(METRICAS_RESIDUAL, index=False)
print('Métricas:', METRICAS_RESIDUAL)
display_metricas(metricas)
con.close()
